## Preparando el ambiente
Instalamos las librerías y configuramos nuestra API Key

In [50]:
import numpy as np
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.messages import HumanMessage, SystemMessage
import os
import ast
from operator import itemgetter
# from elasticsearch import Elasticsearch
import chromadb
from chromadb.config import Settings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.memory.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain.memory import ConversationBufferMemory
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

In [5]:
from huggingface_hub import login
HF_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
login(HF_API_TOKEN)

# Prueba sin RAG
Llamamos el LLM LLama sin ninguna modificación y vemos que alucina o bien desconoce esta información

In [6]:
# Step 1: Define HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-70B-Instruct",
    task="conversational", #THIS MODEL ONLY WORKS WITH CONVERSATIONAL TASK
    temperature=0.7,
    max_new_tokens=512,
)

# Step 2: Initialize ChatHuggingFace
chat = ChatHuggingFace(llm=llm)

# Step 3: Define prompt and parser
prompt = ChatPromptTemplate.from_template("Responde en español: {question}")
parser = StrOutputParser()

# Step 4: Build chain
chain = prompt | chat | parser

In [7]:
response = chain.invoke({"question": "¿Dame mas detalles sobre el destornillador inalambrico de la marca Truper?"})
print(response)

¡Claro! El destornillador inalámbrico de la marca Truper es una herramienta eléctrica portátil y versátil que se utiliza para realizar various tareas de atornillado y desatornillado. A continuación, te proporciono más detalles sobre esta herramienta:

**Características**

* **Potencia**: El destornillador inalámbrico de Truper cuenta con un motor de alta potencia que entrega hasta 18V de energía, lo que lo hace ideal para realizar tareas pesadas y prolongadas.
* **Batería**: La batería es recargable y tiene una capacidad de hasta 2Ah, lo que permite una autonomía de trabajo prolongada. La batería también cuenta con una indicador de carga, lo que te permite monitorear su nivel de carga.
* **Velocidad**: El destornillador tiene dos velocidades: una velocidad lenta de 0-350 rpm y una velocidad rápida de 0-1,300 rpm, lo que te permite adaptarte a diferentes tareas y materiales.
* **Torque**: El destornillador tiene un torque máximo de 30 Nm, lo que lo hace ideal para atornillar y desatorni

# Configurando el RAG

### Los datos 

In [8]:
# WE GET THE DESCRIPTIONS FOR ALL PRODUCTS
import ast 
electric_tool_truper = pd.read_csv("../notebooks/data/sample_electric_truper_products.csv", sep=";")
electric_tool_truper_des = electric_tool_truper["Descripción"].apply(lambda x: ast.literal_eval(x)["descripcion"]).tolist()
df_desc = pd.DataFrame(electric_tool_truper_des, columns=['Text'])
df_desc

,Text
0,"Marca Truper, Línea 18153, Modelo 18153, Tipo ..."
1,"Marca Truper, Modelo LLM-6L, Tipo de producto ..."
2,"Marca Truper, Modelo Torx-7l, Tipo de llave Co..."
3,"Marca Truper, Modelo MAND-7/16, Tipo de mandri..."
4,"Marca Truper, Modelo PICA-X, Tipo de producto ..."
...,...
89,"Marca Truper, Modelo 14182 ""Grata De Copa 3""""..."
90,"Marca Truper, Modelo ALLX-7M, Tipo de llave Al..."
91,"""Marca Truper, Modelo PPC-11R, Tipo de punta C..."
92,"""Marca Truper, Modelo JOY-6, Formato de venta ..."


In [9]:
#In the case where we get more information about a product for now we will embed each paragraph separately

# electric_tool_truper_descriptions = electric_tool_truper_descriptions.apply(lambda x: x.replace("\n", " "))

# electric_tools_document = " ".join(electric_tool_truper_descriptions.tolist())
# document = Document(page_content=electric_tools_document, metadata={"source": "electric_tool_truper"})

"""
## 1. Diviviendo los documentos en chunks
 Los documentos son divididos en pedacitos, o chunks, para que puedan ser correctamente procesados en este contexto.

"""
# text_splitter = RecursiveCharacterTextSplitter(
#                   chunk_size=450,
#                   chunk_overlap=0)


# splits = text_splitter.split_documents(docs)

# for index, split in enumerate(splits):
#   print(f"SPLIT {index + 1}")
#   print(split.page_content)
#   print("--")

# Next the idea will be to save them on a vectorial database

'\n## 1. Diviviendo los documentos en chunks\n Los documentos son divididos en pedacitos, o chunks, para que puedan ser correctamente procesados en este contexto.\n\n'

## Generando los embeddings
Los embeddings son representaciones númericas de la información. Existen varios proveedores de Embeddings, pero vamos a usar los de Hugging Face y para la base de datos vecotial por ahora vamos a utilizar chroma

In [ ]:
# Crear función de embedding
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Crear el vectorstore Chroma
vectorstore = Chroma.from_texts(
    texts=df_desc["Text"].tolist(),
    embedding=embedding_function,
    persist_directory="./chroma_db",  # para almacenamiento persistente
    collection_name="electric_tools_sample"
)
# Crear el retriever que nos va a permitir buscar en el vectorstore
retriever = vectorstore.as_retriever()

## Buscando el chunk más relevante

Notese que nuestro ejercicio entrega sólo un resultado. Sin embargo, veremos ejemplos donde se entregan más resultados de vuelta, por ejemplo los mejores 4.

In [52]:
query = "Dame informacion sobre el destornillador eléctrico inalámbrico de la marca Truper"

# Crear el retriever sin especificar 'collection' en los search_kwargs
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# Obtener el documento más relevante
best_passage = retriever.get_relevant_documents(query)

# Mostrar el contenido del primer resultado
print(best_passage[0].page_content)


Marca Truper, Línea 18153, Modelo 18153, Tipo de producto Destornillador, Tipo de destornillador eléctrico Compacto, Es inalámbrico Sí, Tamaño del mandril 10 mm, Encastre 3/8, Torque máximo 22 Nm, Velocidad mínima de rotación 350 rpm, Velocidad máxima de rotación 1.200 rpm, Accesorios incluidos Batería  Diseño ligero para mayor comodidadDoble engranaje con selector de 2 velocidades mecánicas, botón de dirección de giro y bloqueo del interruptorBroquero de cambio rápido con seguro de retenciónLuz LED para iluminar área de trabajo Indicador de nivel de carga de batería.


## Generando la respuesta con un LLM

In [ ]:
#Modelo Fallback
llm_fallback = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="conversational", #THIS MODEL ONLY WORKS WITH CONVERSATIONAL TASK
    temperature=0.7,
    max_new_tokens=512,
)

chat_fallback = ChatHuggingFace(llm=llm_fallback)

In [49]:
# Memoria de conversación
memory = ConversationBufferMemory(return_messages=True)

# Prompt con contexto incluido en el mensaje del usuario
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente experto en herramientas eléctricas de ferretería. Usa el contexto proporcionado. Si no sabes, di: 'No tengo suficiente información'."),
    ("user", "Contexto: {context}\n\nPregunta: {question}")
])

# Primary y fallback
primary_chain = prompt | chat | StrOutputParser()
fallback_chain = prompt | chat_fallback | StrOutputParser()
main_chain = primary_chain.with_fallbacks([fallback_chain])

# Parallel input: context + query
rag_chain = RunnableParallel({
    "context": itemgetter("question") | retriever,
    "question": RunnablePassthrough()
}) | main_chain


# Para manejar el historial de chat, usamos InMemoryChatMessageHistory
chat_histories = {}

def get_chat_history(session_id: str = "default") -> ChatMessageHistory:
    if session_id not in chat_histories:
        # Aquí usamos InMemory pero bajo la interfaz ChatMessageHistory
        chat_histories[session_id] = InMemoryChatMessageHistory()
    return chat_histories[session_id]

# Composición del chain con historial
rag_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="history"
)

# Función para responder
def answer(query, session_id="default"):
    return rag_with_memory.invoke(
        {"question": query},
        config={"configurable": {"session_id": session_id}}
    )

# Ejemplo de uso
# consulta = "¿Que producto tienen como un taladro inalámbricos de la marca Truper?"
consulta = "Dame información sobre algún destornillador eléctrico inalámbrico de la marca Truper"
respuesta = answer(consulta)
print("🧠 Pregunta:", consulta)
print("💬 Respuesta:", respuesta)

🧠 Pregunta: Dame información sobre algún destornillador eléctrico inalámbrico de la marca Truper
💬 Respuesta: Basándome en la información proporcionada, puedo decirte que la marca Truper ofrece un destornillador eléctrico inalámbrico, específicamente el modelo 18153, que pertenece a la línea 18153. 

Entre sus características destacan:

* Es inalámbrico, lo que ofrece mayor comodidad y libertad de movimiento.
* Tiene un tamaño de mandril de 10 mm y un encastre de 3/8.
* Presenta un torque máximo de 22 Nm y velocidades de rotación que van desde 350 rpm hasta 1.200 rpm.
* Incluye una batería y cuenta con un diseño ligero para mayor comodidad.
* Tiene un doble engranaje con selector de 2 velocidades mecánicas, botón de dirección de giro y bloqueo del interruptor.
* Cuenta con un broquero de cambio rápido con seguro de retención y una luz LED para iluminar el área de trabajo.
* También tiene un indicador de nivel de carga de batería.

Espero que esta información te sea útil. Si necesitas a